# 03 — Train / Validation / Test Split

> **Notebook status: documented reconstruction.** The Dataset V2 split numbers and seed are preserved from the project record. The exact V2 split-generation notebook and complete per-image split manifest were not recovered. An older `Split_FOML_Dataset.ipynb` is retained in `source_material/early_198_image_pilot/`; it belongs to a 198-image pilot and uses seed 42, so it is not the V2 splitter.

## Why three splits?

- **Training set:** used to learn model parameters.
- **Validation set:** used to compare configurations and choose a decision threshold.
- **Test set:** reserved for the final performance estimate and not used to train or tune the model.

Dataset V2 targeted a **70:15:15** split with **random seed 211**. The project record says hash and group checks were included to reduce duplicate leakage.

## Recorded Dataset V2 split

| Split | Real | AI-generated | Total |
|---|---:|---:|---:|
| Training | 437 | 437 | 874 |
| Validation | 94 | 94 | 188 |
| Testing | 94 | 94 | 188 |
| **Total** | **625** | **625** | **1,250** |

The counts above describe the expected split. They are not the actual number loaded during V3 training: Keras skipped WebP files, so the V3 run loaded 872 / 185 / 186 images respectively.


In [ ]:
import pandas as pd

SPLIT_COUNTS = pd.DataFrame([
    {"split":"train", "real":437, "ai_generated":437, "total":874},
    {"split":"validation", "real":94, "ai_generated":94, "total":188},
    {"split":"test", "real":94, "ai_generated":94, "total":188},
])
SPLIT_SEED = 211
TARGET_RATIOS = {"train":0.70, "validation":0.15, "test":0.15}
assert SPLIT_COUNTS[["real", "ai_generated", "total"]].sum().to_dict() == {"real":625, "ai_generated":625, "total":1250}
SPLIT_COUNTS


## Validate a recovered split manifest

When the original `split_manifest.csv` is available, place it at `data/split_manifest.csv`. The expected columns are `split`, `label`, `sha256` and `group_id` (or a source-group column). The checks below verify counts and whether a hash or group occurs in more than one split. The actual V2 manifest is not included in this repository, so the code will report that it is unavailable rather than inventing per-image results.

In [ ]:
from pathlib import Path

manifest_path = Path("data/split_manifest.csv")
if manifest_path.exists():
    split_manifest = pd.read_csv(manifest_path)
    required = {"split", "label", "sha256", "group_id"}
    missing = required - set(split_manifest.columns)
    if missing:
        raise ValueError(f"Split manifest missing columns: {sorted(missing)}")
    actual = (split_manifest.groupby(["split", "label"]).size().unstack(fill_value=0))
    display(actual)
    for key in ["sha256", "group_id"]:
        cross_split = split_manifest.groupby(key)["split"].nunique()
        leaked = cross_split[cross_split > 1]
        print(f"{key} values crossing splits:", len(leaked))
        assert len(leaked) == 0, f"{key} leakage detected across splits"
else:
    print("No Dataset V2 per-image split manifest was recovered; only the published split counts can be checked here.")


## Example of a new group-aware split (not the historical split)

If the original manifest is lost but the source images and stable grouping metadata are available, this example demonstrates a two-stage `GroupShuffleSplit`. It uses seed 211 and keeps groups intact. Because `GroupShuffleSplit` does not stratify classes, the resulting class counts must be inspected and may not match 437/437 and 94/94. **Do not overwrite the historical split with this output or claim that it reproduces the original file assignment.**

In [ ]:
def make_new_group_aware_split(frame: pd.DataFrame, seed: int = 211) -> pd.DataFrame:
    from sklearn.model_selection import GroupShuffleSplit
    required = {"path", "label", "group_id"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Input metadata missing: {sorted(missing)}")
    first = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=seed)
    train_idx, temp_idx = next(first.split(frame, y=frame["label"], groups=frame["group_id"]))
    train = frame.iloc[train_idx].copy(); temp = frame.iloc[temp_idx].copy()
    second = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed)
    val_rel, test_rel = next(second.split(temp, y=temp["label"], groups=temp["group_id"]))
    train["split"] = "train"
    temp_val = temp.iloc[val_rel].copy(); temp_val["split"] = "validation"
    temp_test = temp.iloc[test_rel].copy(); temp_test["split"] = "test"
    return pd.concat([train, temp_val, temp_test], ignore_index=True)

# To create a separate NEW split, load a manifest with path, label and group_id.
# new_split = make_new_group_aware_split(pd.read_csv("data/source_manifest.csv"))
# display(new_split.groupby(["split", "label"]).size().unstack(fill_value=0))


## V3 loader coverage

| Split | Expected images | Loaded by V3 training run | Skipped WebP images |
|---|---:|---:|---:|
| Training | 874 | 872 | 2 |
| Validation | 188 | 185 | 3 |
| Testing | 188 | 186 | 2 |

The seven skipped files are a documented limitation. The V3 model metrics must be described using the effective loaded counts, not the expected split totals. The V3 evaluation also was not a held-out-generator evaluation; source groups overlap across splits.